# DQL Aggregate Functions: MAX and MIN

`MAX` returns the largest value, and `MIN` returns the smallest value.


## How to Think About Aggregates

Aggregate functions summarize many rows into fewer values. They answer questions such as how many, how much, what average, what highest, and what lowest.

There are two common patterns:

* Ungrouped aggregates summarize the whole DataFrame.
* Grouped aggregates summarize each category, such as each department or job.

When reading aggregate output, check whether nulls are ignored, whether the grouping level is correct, and whether the output column names are clear.

## CSV Data Source

CSV stores data as plain text rows. The loader creates the table names used by the examples: `emp`, `dept`, `salgrade`, `projects`, and `emp_projects`.


In [ ]:
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName("pyspark-sql-tutorial")
    .getOrCreate()
)

candidate_dirs = [
    Path.cwd() / "data",
    Path.cwd().parent / "data",
    Path.cwd(),
    Path.cwd().parent,
]

DATA_DIR = next((path for path in candidate_dirs if (path / "emp.csv").exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError("Could not find emp.csv. Put the CSV files in ./data or the current folder.")

print(f"Reading CSV files from: {DATA_DIR}")

emp = spark.read.option("header", True).option("inferSchema", True).csv(str(DATA_DIR / "emp.csv"))
dept = spark.read.option("header", True).option("inferSchema", True).csv(str(DATA_DIR / "dept.csv"))
salgrade = spark.read.option("header", True).option("inferSchema", True).csv(str(DATA_DIR / "salgrade.csv"))
projects = spark.read.option("header", True).option("inferSchema", True).csv(str(DATA_DIR / "projects.csv"))
emp_projects = spark.read.option("header", True).option("inferSchema", True).csv(str(DATA_DIR / "emp_projects.csv"))

for name, df in {
    "emp": emp,
    "dept": dept,
    "salgrade": salgrade,
    "projects": projects,
    "emp_projects": emp_projects,
}.items():
    df.createOrReplaceTempView(name)


In [ ]:
emp.show(5)
dept.show()
salgrade.show()
projects.show()
emp_projects.show(5)


## Parquet Data Source

Parquet stores data in a compressed columnar format. The same table names are created, so the DQL examples work the same way after loading Parquet.


In [ ]:
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName("pyspark-parquet-sql-tutorial")
    .getOrCreate()
)

candidate_dirs = [
    Path.cwd() / "data",
    Path.cwd().parent / "data",
    Path.cwd(),
    Path.cwd().parent,
]

DATA_DIR = next((path for path in candidate_dirs if (path / "dept.parquet").exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError("Could not find dept.parquet. Put the Parquet files in ./data or the current folder.")

emp_paths = sorted(DATA_DIR.glob("emp_part_*.parquet"))
if not emp_paths:
    raise FileNotFoundError("Could not find emp_part_*.parquet files in the Parquet data folder.")

print(f"Reading Parquet files from: {DATA_DIR}")

emp = spark.read.parquet(*[str(path) for path in emp_paths])
dept = spark.read.parquet(str(DATA_DIR / "dept.parquet"))
salgrade = spark.read.parquet(str(DATA_DIR / "salgrade.parquet"))
projects = spark.read.parquet(str(DATA_DIR / "projects.parquet"))
emp_projects = spark.read.parquet(str(DATA_DIR / "emp_projects.parquet"))

for name, df in {
    "emp": emp,
    "dept": dept,
    "salgrade": salgrade,
    "projects": projects,
    "emp_projects": emp_projects,
}.items():
    df.createOrReplaceTempView(name)


In [ ]:
emp.show(5)
dept.show()
salgrade.show()
projects.show()
emp_projects.show(5)


## MAX and MIN

Use `max` and `min` to find extremes.

`MAX` finds the largest value. `MIN` finds the smallest value.

These work on numbers, dates, and strings, but the meaning depends on the data type.


In [ ]:
emp.agg(
    F.max("sal").alias("highest_salary"),
    F.min("sal").alias("lowest_salary"),
    F.min("hiredate").alias("first_hire_date"),
    F.max("hiredate").alias("last_hire_date")
).show()

spark.sql("""
SELECT MAX(sal) AS highest_salary,
       MIN(sal) AS lowest_salary,
       MIN(hiredate) AS first_hire_date,
       MAX(hiredate) AS last_hire_date
FROM emp
""").show()
